In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
import re

In [0]:
storage_account = "startupvillagedatalake"
scope_name = "kv-startupvillage"   

client_id = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope=scope_name, key="tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
dbutils.fs.ls("abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/")

[FileInfo(path='abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/', name='ingestion_date=2026-04-12/', size=0, modificationTime=1776009797000)]

In [0]:
landing_root = "abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi"
bronze_root = "abfss://bronze@startupvillagedatalake.dfs.core.windows.net/glpi"

entity = "reserveditems"

landing_entity_root = f"{landing_root}/{entity}"
bronze_entity_root = f"{bronze_root}/{entity}"

In [0]:
def list_ingestion_dates(path: str) -> set[str]:
    """
    Lists partitions like .../ingestion_date=YYYY-MM-DD/ under `path`.
    Returns a set of 'YYYY-MM-DD' strings.
    """
    dates = set()
    for fi in dbutils.fs.ls(path):
        m = re.search(r"ingestion_date=([0-9]{4}-[0-9]{2}-[0-9]{2})", fi.path)
        if m:
            dates.add(m.group(1))
    return dates

def get_missing_dates(landing_path: str, bronze_path: str) -> list[str]:
    landing_dates = list_ingestion_dates(landing_path)

    try:
        bronze_dates = list_ingestion_dates(bronze_path)
    except Exception:
        # First run: bronze path may not exist yet
        bronze_dates = set()

    missing = sorted(landing_dates - bronze_dates)  # ascending => older first
    return missing

In [0]:
missing_dates = get_missing_dates(landing_entity_root, bronze_entity_root)

print(f"[{entity}] Landing partitions:", sorted(list_ingestion_dates(landing_entity_root)))
print(f"[{entity}] Bronze partitions:", sorted(list_ingestion_dates(bronze_entity_root)) if dbutils.fs.ls(bronze_root) else [])
print(f"[{entity}] Missing partitions to process:", missing_dates)

[reserveditems] Landing partitions: ['2026-04-12']
[reserveditems] Bronze partitions: []
[reserveditems] Missing partitions to process: ['2026-04-12']


In [0]:
if not missing_dates:
    print(f"[{entity}] No new ingestion_date partitions to process. Exiting.")
else:
    for d in missing_dates:
        src = f"{landing_entity_root}/ingestion_date={d}/"
        print(f"[{entity}] Processing ingestion_date={d} from {src}")

        df = (
            spark.read.option("multiline", "false")
            .json(src)
            .withColumn("ingestion_date", lit(d).cast("date"))
            .withColumn("_source_file", input_file_name())
            .withColumn("_source_system", lit("glpi"))
            .withColumn("_ingest_bronze_ts", current_timestamp())
        )

        # Idempotent write: overwrite only this partition
        (
           df.write.format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"ingestion_date = '{d}'")
            .partitionBy("ingestion_date")
            .save(bronze_entity_root)
        )

        print(f"[{entity}] Wrote {df.count()} rows to Bronze for ingestion_date={d}")

[reserveditems] Processing ingestion_date=2026-04-12 from abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/
[reserveditems] Wrote 12 rows to Bronze for ingestion_date=2026-04-12


In [0]:
df.display()

comment,entities_id,id,is_active,is_deleted,is_recursive,items_id,itemtype,links,ingestion_date,_source_file,_source_system,_ingest_bronze_ts
null,Root entity,1,1,0,0,Salle COSY (S-4),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/1, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,2,1,0,0,Salle Panorama (S-12),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/2, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,3,1,0,0,Salle Confidentielle (S-24),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/5, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,4,1,0,0,Salle Avant Première (S15),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/4, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,5,1,0,0,Salle Transparente (S-13),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/3, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,6,1,0,0,Meeting Room (S-23),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/6, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,7,0,0,0,Corner Room (S-20),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/7, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,8,1,0,0,Salle Arabesque (S-5),PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/8, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,12,1,0,0,Salle 11,PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/112, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/glpi/reserveditems/ingestion_date=2026-04-12/reserveditem_0_99.json,glpi,2026-04-12T16:42:05.59Z
null,Root entity,13,1,0,0,Salle 10,PluginRoomRoom,"List(List(http://inventaire.startupvillage.tn/apirest.php/Entity/0, Entity), List(http://inventaire.startupvillage.tn/apirest.php/PluginRoomRoom/113, PluginRoomRoom))",2026-04-12,abfss://landing@startupvillagedatalake.dfs.core.windows.net/